# 4. Installation et configuration de YOLOv9 (dépôt officiel `WongKinYiu/yolov9`)

Ce notebook vérifie l'installation locale du dépôt officiel [WongKinYiu/yolov9](https://github.com/WongKinYiu/yolov9), les fichiers clés (`train_dual.py`, `models/detect/yolov9-c.yaml`, `data/hyps/hyp.scratch-high.yaml`, `yolov9-c.pt`), la compatibilité avec `data_1.yaml` et le mécanisme de **Transfer Learning**.

In [ ]:
from pathlib import Path
import sys
import yaml
import torch

YOLO_DIR = Path("../yolov9").resolve()
DATA_YAML = Path("../trafic_data/data_1.yaml").resolve()
TRAIN_SCRIPT = YOLO_DIR / "train_dual.py"
CFG_YAML = YOLO_DIR / "models/detect/yolov9-c.yaml"
HYP_YAML = YOLO_DIR / "data/hyps/hyp.scratch-high.yaml"
WEIGHTS_PT = YOLO_DIR / "yolov9-c.pt"

for label, path in [
    ("Dépôt YOLOv9", YOLO_DIR),
    ("Script train_dual.py", TRAIN_SCRIPT),
    ("Architecture yolov9-c.yaml", CFG_YAML),
    ("Hyperparamètres hyp.scratch-high.yaml", HYP_YAML),
    ("Poids yolov9-c.pt", WEIGHTS_PT),
    ("Dataset data_1.yaml", DATA_YAML),
]:
    assert path.exists(), f"Introuvable : {path}"
    size_info = f"({path.stat().st_size / (1024*1024):.2f} Mo)" if path.is_file() else "(dossier)"
    print(f"[OK] {label:38s} -> {path} {size_info}")

In [ ]:
# Inspection des fichiers de configuration : architecture (yolov9-c.yaml), hyperparamètres et dataset
cfg_model = yaml.safe_load(CFG_YAML.read_text(encoding="utf-8"))
hyp = yaml.safe_load(HYP_YAML.read_text(encoding="utf-8"))

print(f"yolov9-c.yaml -> nc={cfg_model['nc']}, depth_multiple={cfg_model['depth_multiple']}, width_multiple={cfg_model['width_multiple']}")
print(f"  Couches backbone : {len(cfg_model['backbone'])} | Couches head : {len(cfg_model['head'])} | Tête finale : {cfg_model['head'][-1][2]}")
print(f"hyp.scratch-high.yaml -> lr0={hyp['lr0']}, lrf={hyp['lrf']}, momentum={hyp['momentum']}, weight_decay={hyp['weight_decay']}, mosaic={hyp['mosaic']}, mixup={hyp['mixup']}")

if str(YOLO_DIR) not in sys.path:
    sys.path.insert(0, str(YOLO_DIR))

from utils.general import check_dataset, intersect_dicts
from models.yolo import Model

data_dict = check_dataset(str(DATA_YAML))
print(f"data_1.yaml validé -> nc={data_dict['nc']}, train={data_dict['train']}, val={data_dict['val']}, test={data_dict['test']}")

In [ ]:
# Vérification du Transfer Learning : chargement des poids yolov9-c.pt (80 classes COCO) vers notre modèle (21 classes)
ckpt = torch.load(WEIGHTS_PT, map_location="cpu", weights_only=False)
model = Model(str(CFG_YAML), ch=3, nc=int(data_dict["nc"]), anchors=hyp.get("anchors"))

csd = ckpt["model"].float().state_dict()
csd_matched = intersect_dicts(csd, model.state_dict(), exclude=["anchor"])
model.load_state_dict(csd_matched, strict=False)

total_params = sum(p.numel() for p in model.parameters())
print(f"Tenseurs transférés par Transfer Learning : {len(csd_matched)}/{len(model.state_dict())}")
print(f"Paramètres du modèle YOLOv9-C (nc={data_dict['nc']}) : {total_params:,}")

### Concepts clés : Transfer Learning et séparation des composants YOLOv9

1. **Rôle des poids préentraînés et principe du Transfer Learning :**
   - `yolov9-c.pt` contient les poids déjà entraînés sur **MS COCO (80 classes)**.
   - Les premières couches (backbone + partie du neck) savent déjà extraire des caractéristiques visuelles universelles (bords, textures, roues, formes de véhicules).
   - Lors du *fine-tuning* sur nos **21 classes de véhicules**, `train_dual.py` copie automatiquement tous les tenseurs compatibles (`intersect_dicts`) et réinitialise uniquement les couches de sortie dont la dimension dépend du nombre de classes (`nc = 21` au lieu de `80`). Cela accélère fortement la convergence et améliore la généralisation sur un dataset de taille modérée (`2 297` images).

2. **Différences entre les composants :**
   - **Architecture du modèle (`models/detect/yolov9-c.yaml`) :** plan structurel du réseau (backbone GELAN/RepNCSPELAN4, branche auxiliaire PGI, tête `DualDDetect`, `nc`). Elle ne contient **aucun poids appris**.
   - **Poids préentraînés (`yolov9-c.pt`) :** valeurs numériques des paramètres (*state_dict*) apprises sur COCO, servant de point de départ au Transfer Learning.
   - **Hyperparamètres (`data/hyps/hyp.scratch-high.yaml`) :** réglages d'optimisation et d'augmentation de données (`lr0`, `momentum`, `weight_decay`, pondérations de pertes `box`/`cls`/`dfl`, `mosaic`, `mixup`, `copy_paste`).
   - **Script d'entraînement (`train_dual.py`) :** orchestrateur PyTorch qui charge le dataset (`data_1.yaml`), construit le modèle (`--cfg`), transfère les poids (`--weights`), applique les hyperparamètres (`--hyp`), calcule la perte duale PGI (`ComputeLoss` dans `utils/loss_tal_dual.py`) et sauvegarde `best.pt` / `last.pt`.